# 포켓몬 30주년 한·중·일 팬덤 소비자 비교 — 수집 → 분류 → 시트 채우기 → 분석 (올인원)

엑셀 시트에 들어 있는 **링크(「수집 채널 링크」 + 홈페이지 예시 URL)** 에서 정보를 모으고, 시트 기준대로 자동 분류한 뒤, **원본 xlsx 복사본에 채워 넣고**, 같은 데이터로 **표 3개 + 그래프 10개**까지 한 번에 만듭니다. (`Run All`)

| 단계 | 하는 일 | 결과물 |
|---|---|---|
| ① 수집 | 홈페이지 크롤링 · YouTube(공식 API) · 빌리빌리(yt-dlp, 실험적) · `manual/` CSV(인스타·X·웨이보) | 메모리 |
| ② 분류 | 캠페인 목적 · 제안 4종 · 메시지 유형 · 감성 · 행동의도 · 토픽 (사전/모델 기반 1차 코딩) | `data/*.csv` |
| ③ 시트 채우기 | 원본 6개 xlsx를 복사해 4행부터 기입 (원본 불변) | `수집결과/*.xlsx` |
| ④ 분석 | 데이터 현황 · 핵심 지표 · 통계 검정 표 + 그래프 10개 | `결과/` |
| ⑤ 검수 표본 | 사람 검수용 표본 + Cohen's κ 계산 함수 | `data/review_sample_*.csv` |

**준비**
```
pip install requests beautifulsoup4 pandas openpyxl matplotlib scipy
pip install yt-dlp                    # (선택) 빌리빌리
pip install transformers torch        # (선택) 다국어 감성 모델
pip install playwright && playwright install chromium   # (선택) JS 렌더링 홈페이지
```
**YouTube API 키**: Google Cloud Console → YouTube Data API v3 사용 설정 → API 키 → 환경변수 `YOUTUBE_API_KEY` 또는 아래 설정 셀에 입력
**원본 xlsx 6개**를 이 노트북과 같은 폴더에 두세요.

**자동 수집이 안 되는 곳**: 인스타그램 · X · 웨이보 — 공식 API가 막혀 있거나 유료이고, 로그인 필요 페이지의 자동 크롤링은 약관 위반 소지가 큽니다. `manual/` 폴더에 자동 생성되는 템플릿 CSV에 게시물 정보만 붙여넣고 다시 실행하면 같은 파이프라인에 합쳐집니다.

> ⚠ 분류는 **1차 자동 코딩**입니다(비고 칸에 표시). 논문에는 ⑤의 검수 표본으로 일치도(κ)를 함께 보고하세요. 분석만 다시 돌리려면 **설정 셀 → '④ 분석' 섹션**만 실행하면 됩니다(`data/*.csv`를 읽음).

In [ ]:
import os, re, time, json, shutil, unicodedata, random, warnings
from collections import Counter
from datetime import date
from urllib import robotparser
from urllib.parse import urljoin, urlparse

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
import openpyxl
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE

warnings.filterwarnings("ignore")

# ───────── 설정 ─────────
DATA_DIR      = "."            # 원본 xlsx 6개가 있는 폴더 (읽기만 함)
CSV_DIR       = "data"         # 분류 결과 CSV
XLSX_OUT_DIR  = "수집결과"      # 채워진 xlsx 사본
MANUAL_DIR    = "manual"       # 인스타그램·X·웨이보 등 수동 CSV
for d_ in (CSV_DIR, XLSX_OUT_DIR, MANUAL_DIR): os.makedirs(d_, exist_ok=True)

YOUTUBE_API_KEY = os.environ["GOOGLE_API_KEY"]
START_DATE = "2026-01-01"      # 수집 시작일
START_DATE = "2026-01-01"      # 수집 시작일
END_DATE   = None              # None이면 오늘까지
MAX_POSTS_PER_CHANNEL = 100    # 채널당 최대 영상 수
COMMENTS_PER_POST     = 30     # 영상당 최대 댓글 수 (0이면 댓글 수집 안 함)
COMMENT_ORDER = "relevance"    # "relevance"(인기순) / "time"(최신순) ※ 표집 방식으로 논문에 명시
ONLY_30TH_KEYWORD = False      # True면 제목·설명에 30周年/30주년/30th 가 있는 영상만

HOMEPAGE_EXTRA = {"한국": [], "중국": [], "일본": []}   # 홈페이지 URL을 더 넣으려면 여기에
CRAWL_DEPTH = 1                # 시작 URL에서 같은 도메인 링크를 몇 단계까지 따라갈지
MAX_PAGES_PER_SITE = 15
USE_PLAYWRIGHT = False         # JS로 그려지는 사이트면 True
REQUEST_DELAY = 1.5            # 요청 간 지연(초)
UA = {"User-Agent": "Mozilla/5.0 (academic-research crawler; contact: your-email@example.com)"}   # ← 본인 이메일로

USE_SENT_MODEL = True          # 다국어 감성 모델 사용(미설치/오프라인이면 사전 기반으로 자동 대체)
WRITE_SHEETS   = True          # False면 xlsx 채우기 생략

ORDER = ["한국", "중국", "일본"]
CC = {"한국": "KR", "중국": "CN", "일본": "JP"}
START_TS = pd.Timestamp(START_DATE)
END_TS = pd.Timestamp(END_DATE) if END_DATE else pd.Timestamp.today().normalize()
TODAY = date.today().strftime("%Y-%m-%d")

# ① 수집

## 1. 자동 분류 규칙 (사전 기반 · 수정 가능)

- **캠페인 목적**: 상품판매 / 행사참여 / 브랜드홍보 — 키워드 점수 최대값
- **제안 4종**(수집·배틀·추억·행사): 키워드 포함 시 O
- **메시지 유형**(내부자형 / 초대형): ⚠ **연구의 정의에 맞게 사전을 조정하세요.** 기본값은 *코어팬만 아는 용어(카드 레어도·게임 시스템·세대/개체값 용어 등)가 많으면 내부자형, 대중 호명(모두·가족·전 세계 등)이 많으면 초대형*으로 가정한 것입니다.
- **감성**: 다국어 모델(`cardiffnlp/twitter-xlm-roberta-base-sentiment`, 선택) → 없으면 사전 점수로 대체
- **행동의도**: 구매·예약·방문 의사 키워드, **토픽**: 사전 기반 다중 태깅
- 모든 사전은 한·중·일·영 키워드를 함께 담았으므로 국가별 차이는 사전 확장으로 보완하세요.

In [ ]:
def _kw(*groups):
    return [k.lower() for g in groups for k in g.split("|") if k]

PURPOSE_KW = {
 "상품판매": _kw("예약|구매|판매|발매|출시|한정|굿즈|부스터|박스|상품|스토어|샵|프로모|특전|재입고|사전예약",
                 "予約|発売|販売|購入|限定|グッズ|商品|ストア|特典|再販|ポケセン|ポケモンセンター",
                 "预约|预售|发售|购买|限定|周边|商品|官方商店|特典|补货|旗舰店",
                 "pre-order|preorder|buy|shop|store|on sale|release|launch|limited|merch"),
 "행사참여": _kw("이벤트|행사|팝업|페스티벌|참가|응모|추첨|대회|전시|체험|캠페인|개최|투어|데이아웃|나이트아웃",
                 "イベント|開催|応募|キャンペーン|参加|抽選|大会|展|ポップアップ|フェス|会場|ツアー",
                 "活动|参加|抽奖|展览|快闪|大赛|线下|报名|巡展|现场|征集",
                 "event|campaign|popup|pop-up|festival|tournament|giveaway|lottery|day out|night out|exhibit"),
 "브랜드홍보": _kw("30년|30주년|함께|모험|감사|영상|스페셜|역사|기념|스토리|메시지|로고|무비",
                   "30周年|30年|ありがとう|一緒|冒険|歴代|スペシャル|ムービー|記念|ロゴ|メッセージ",
                   "30周年|三十|一起|冒险|感谢|回顾|纪念|特别|短片|寄语",
                   "anniversary|thank|journey|adventure|together|special movie|celebrat"),
}
PROPOSAL_KW = {
 "p_collect": _kw("수집|도감|컬렉션|컴플리트|모으|콜렉|완성", "収集|コレクション|図鑑|コンプリート|集め|コンプ", "收集|图鉴|收藏|集卡|集换|全图鉴|集齐", "collect|complete|pokedex|pokédex|set"),
 "p_battle":  _kw("배틀|대전|승부|토너먼트|랭크|대회|챔피언|리그", "バトル|対戦|勝負|トーナメント|大会|ランクマ|チャンピオン|リーグ", "对战|对决|竞技|锦标赛|天梯|大赛|冠军|联赛", "battle|tournament|versus|ranked|championship|league"),
 "p_memory":  _kw("추억|그때|어릴|처음|그리운|복각|레트로|옛|초대|1세대|향수", "思い出|懐かし|あの頃|初代|復刻|レトロ|歴代|初めて", "回忆|怀旧|童年|初代|复刻|经典|情怀|爷青回", "nostalg|memory|memories|retro|classic|throwback|remember|childhood|original|first generation"),
 "p_event":   _kw("행사|이벤트|팝업|페스티벌|전시|오프라인|현장|개최|투어", "イベント|開催|ポップアップ|フェス|展|会場|ツアー", "活动|线下|快闪|展览|现场|巡展|开幕", "event|pop-up|popup|festival|exhibit|day out|night out|tour"),
}
INSIDER_KW = _kw("레어|SAR|SR|UR|AR|ex카드|에이스스펙|테라스탈|메가진화|다이맥스|종족값|개체값|노력치|특성|배틀타워|레이드|이로치|색다른|오리진|하이퍼볼|부스터팩|확장팩|프로모카드|PSA|그레이딩",
                 "レア|SAR|SR|UR|ARカード|エースペック|テラスタル|メガシンカ|ダイマックス|種族値|個体値|努力値|特性|色違い|厳選|レイド|拡張パック|プロモカード|PSA|鑑定",
                 "稀有|SAR|SR|UR|AR卡|王牌|太晶|超级进化|极巨化|种族值|个体值|努力值|特性|异色|闪光|孵蛋|团战|扩展包|促销卡|评级|PSA",
                 "rarity|illustration rare|special illustration|ace spec|terastal|mega evolution|dynamax|base stats|ivs|evs|shiny|raid|expansion|promo card|grading")
BROAD_KW = _kw("모두|누구나|함께|가족|전 세계|온 세상|모든 트레이너|여러분|세대를 넘어", "みんな|誰でも|みなさん|家族|世界中|全世界|世代を超え|みんなで", "大家|所有人|每个人|全世界|全球|一家|跨越世代|各位", "everyone|everybody|all trainers|world|family|all ages|fans around")

INTENT_RE = re.compile("|".join(_kw(
 "구매|살래|사러|예약|질렀|지름|사야|갖고싶|가지고싶|가고싶|가자|참가|응모|입덕|시작할|해볼|뽑으러|박스깡|사고싶|주문",
 "買う|買いたい|買います|予約|欲しい|ほしい|ゲット|行きたい|行く|ポチ|応募|参加|始め|買った",
 "买|想买|入手|预约|抢|想要|冲|下单|剁手|想去|报名|参加|开箱|安排|蹲",
 "buy|preorder|pre-order|want|need to get|gonna get|take my money|can't wait|cant wait|sign up|join")))

POS_KW = _kw("최고|좋|감동|사랑|대박|귀엽|귀여|예쁘|멋|기대|축하|감사|행복|레전드|갓|소름|눈물|벅차|고마워|미쳤|미친",
             "最高|好き|可愛い|かわいい|感動|嬉しい|ありがとう|楽しみ|素敵|おめでとう|尊い|エモい|泣|神|うれしい",
             "喜欢|可爱|感动|太好了|期待|恭喜|爱了|好看|神作|泪目|爷青回|yyds|绝了|太棒|感谢|泪",
             "awesome|love|great|amazing|cute|beautiful|congrat|thank|nostalg|hype|wholesome|best|❤|😍|🥰|👍|🎉|🥹")
NEG_KW = _kw("별로|실망|최악|비싸|거품|쓰레기|불만|품절|사기|짜증|노잼|아쉽|돈낭비|환불|폭리|스캘|되팔|논란|화나",
             "最悪|がっかり|高い|ひどい|転売|つまらない|残念|ぼったくり|詐欺|不満|炎上|買えない|売り切れ",
             "失望|垃圾|太贵|坑|黄牛|差评|无聊|难看|骗|恶心|割韭菜|退钱|抢不到|缺货|炒",
             "disappoint|worst|expensive|scalp|terrible|boring|trash|scam|rip-off|ripoff|overpriced|hate|👎|😡|🤮")

TOPIC_KW = {
 "추억·향수": _kw("추억|향수|어릴|옛날|그때|레트로", "思い出|懐かし|あの頃|子供の頃|レトロ", "回忆|怀旧|童年|爷青回|情怀", "nostalg|childhood|memories|retro"),
 "일러스트·아트": _kw("일러스트|그림|작화|아트|디자인", "イラスト|絵|作画|アート|デザイン", "插画|画风|作画|美术|设计", "illustration|artwork|art style|design"),
 "영상·연출": _kw("영상|연출|뮤비|무비|bgm|애니", "映像|演出|ムービー|アニメ|MV", "视频|演出|短片|动画|MV|bgm", "video|animation|trailer|movie|mv|cinematic"),
 "가격·가성비": _kw("가격|비싸|가성비|물가|거품|원|할인", "価格|値段|高い|安い|円|割引", "价格|太贵|性价比|涨价|元|折扣", "price|expensive|cheap|overpriced|discount|worth"),
 "굿즈·상품": _kw("굿즈|상품|피규어|인형|한정|특전", "グッズ|商品|フィギュア|ぬいぐるみ|限定|特典", "周边|商品|手办|玩偶|限定|特典", "merch|goods|figure|plush|limited"),
 "카드·팩": _kw("카드|팩|박스|부스터|레어|sar|개봉", "カード|パック|ボックス|BOX|レア|SAR|開封", "卡牌|卡包|补充包|稀有|开箱|集换", "card|pack|box|booster|rare|pull|opening"),
 "게임": _kw("게임|스위치|신작|리메이크|배틀|도감", "ゲーム|スイッチ|新作|リメイク|対戦|図鑑", "游戏|switch|新作|重制|对战|图鉴", "game|switch|remake|battle|pokedex"),
 "팝업·오프라인": _kw("팝업|오프라인|전시|행사|현장|매장|웨이팅", "ポップアップ|会場|展|イベント|店舗|行列", "快闪|线下|展览|现场|门店|排队", "pop-up|popup|event|venue|queue|store"),
 "음악·OST": _kw("음악|ost|노래|bgm|브금", "音楽|曲|BGM|OST|歌", "音乐|ost|歌|bgm|配乐", "music|ost|song|bgm|soundtrack"),
 "캐릭터": _kw("피카츄|이브이|리자몽|캐릭터|최애", "ピカチュウ|イーブイ|リザードン|キャラ|推し", "皮卡丘|伊布|喷火龙|角色|本命", "pikachu|eevee|charizard|character|favorite"),
 "품절·리셀": _kw("품절|재고|리셀|되팔|스캘퍼|구하기", "売り切れ|在庫|転売|抽選|買えない", "断货|缺货|黄牛|炒价|抢不到|限购", "sold out|stock|scalper|resell|restock"),
}

def _count(text, kws):
    t = str(text).lower()
    return sum(t.count(k) for k in kws)

def classify_purpose(text):
    sc = {k: _count(text, v) * (0.6 if k == "브랜드홍보" else 1.0) for k, v in PURPOSE_KW.items()}
    top = sorted(sc.items(), key=lambda x: -x[1])
    if top[0][1] == 0:
        return "브랜드홍보", True                       # 단서 없음 → 기본값 + 검수 대상
    return top[0][0], (top[0][1] - top[1][1]) < 1      # 1위-2위 차이가 작으면 검수 대상

def classify_proposals(text):
    return {k: ("O" if _count(text, v) > 0 else "X") for k, v in PROPOSAL_KW.items()}

def classify_msg_type(text):
    ins, brd = _count(text, INSIDER_KW), _count(text, BROAD_KW)
    return ("내부자형" if ins > brd else "초대형"), (ins == brd)

def classify_intent(text):
    return "O" if INTENT_RE.search(str(text).lower()) else "X"

def classify_topics(text, k=3):
    sc = {t: _count(text, v) for t, v in TOPIC_KW.items()}
    hit = [t for t, c in sorted(sc.items(), key=lambda x: -x[1]) if c > 0][:k]
    return " / ".join(hit) if hit else "기타"

_SENT_PIPE = None
def _load_sent_model():
    global _SENT_PIPE
    if _SENT_PIPE is None and USE_SENT_MODEL:
        try:
            from transformers import pipeline
            _SENT_PIPE = pipeline("text-classification", model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
                                  truncation=True, max_length=256)
            print("감성 모델 로드 완료")
        except Exception as e:
            print("감성 모델을 쓸 수 없어 사전 기반으로 대체합니다:", str(e)[:80])
            _SENT_PIPE = False
    return _SENT_PIPE

def sentiment_lexicon(text):
    p, n = _count(text, POS_KW), _count(text, NEG_KW)
    return 1 if p > n else (-1 if n > p else 0)

def classify_sentiment_batch(texts):
    texts = [str(t) for t in texts]
    pipe = _load_sent_model()
    if pipe:
        out = []
        for i in range(0, len(texts), 32):
            for r in pipe(texts[i:i + 32]):
                out.append({"positive": 1, "neutral": 0, "negative": -1}.get(r["label"].lower(), 0))
        return out
    return [sentiment_lexicon(t) for t in texts]

# ── 간단 점검 ──
print(classify_purpose("30주년 기념 카드 사전예약 시작! 한정 굿즈"), classify_proposals("복각 카드 컬렉션 대회 개최"),
      classify_msg_type("SAR 레어 카드 개체값"), classify_intent("이거 사러 간다"), classify_topics("일러스트 미쳤다 가격은 비싸다"))

## 2. 채널·URL 읽기 (시트에서 자동)

In [ ]:
def nfc(s): return unicodedata.normalize("NFC", str(s))

def find_file(country, keyword):
    for fn in sorted(os.listdir(DATA_DIR)):
        n = nfc(fn)
        if n.lower().endswith(".xlsx") and not n.startswith("~$") and n.startswith(country) and keyword in n:
            return os.path.join(DATA_DIR, fn)
    raise FileNotFoundError(f"'{country}' + '{keyword}' 파일을 {os.path.abspath(DATA_DIR)} 에서 찾지 못했습니다.")

def norm_platform(x):
    s = str(x).strip().lower()
    if s in ("", "nan"): return np.nan
    if any(k in s for k in ["유튜브", "youtube", "油管"]): return "YouTube"
    if any(k in s for k in ["인스타", "instagram", "insta"]): return "Instagram"
    if any(k in s for k in ["웨이보", "weibo", "微博"]): return "Weibo"
    if any(k in s for k in ["빌리빌리", "bilibili", "哔哩", "b站"]): return "Bilibili"
    if s == "x" or "트위터" in s or "twitter" in s or s.startswith("x(") or s.startswith("x ("): return "X"
    return str(x).strip()

def read_channels(country):
    sheets = pd.read_excel(find_file(country, "SNS"), sheet_name=None, dtype=object)
    name = next(n for n in sheets if "채널" in nfc(n))
    df = sheets[name].dropna(how="all").iloc[:, :5].copy()
    df.columns = ["구분", "channel_type", "channel_name", "url", "note"]
    df = df[df["url"].notna()].copy()
    df["url"] = df["url"].astype(str).str.strip().map(lambda u: u if u.startswith("http") else "https://" + u)
    df["platform"] = df["구분"].map(norm_platform)
    df["country"] = country
    return df.reset_index(drop=True)

def read_homepage_urls(country):
    df = pd.read_excel(find_file(country, "홈페이지"), header=0, dtype=object)
    urls = []
    if len(df) > 1 and pd.notna(df.iloc[1, 13]) and str(df.iloc[1, 13]).startswith("http"):    # 3행 작성예시의 URL
        urls.append(str(df.iloc[1, 13]).strip())
    urls += HOMEPAGE_EXTRA.get(country, [])
    return list(dict.fromkeys(urls))

channels = pd.concat([read_channels(c) for c in ORDER], ignore_index=True)
hp_urls = {c: read_homepage_urls(c) for c in ORDER}
print(channels[["country", "platform", "channel_type", "channel_name", "url"]].to_string())
print(hp_urls)

In [ ]:
channels = pd.concat([read_channels(c) for c in ORDER], ignore_index=True)
hp_urls = {c: read_homepage_urls(c) for c in ORDER}
print(channels[["country", "platform", "channel_type", "channel_name", "url"]].to_string())
print("\n홈페이지 시작 URL:", hp_urls)

## 3. YouTube 수집 (공식 Data API v3)

채널의 업로드 목록 → 기간 필터 → 조회수·좋아요·댓글수(유튜브는 공유수 미제공 → 0) → 영상별 댓글 N개

In [ ]:
YT = "https://www.googleapis.com/youtube/v3"

def yt_get(endpoint, **params):
    params["key"] = YOUTUBE_API_KEY
    for attempt in range(4):
        r = requests.get(f"{YT}/{endpoint}", params=params, timeout=30)
        if r.status_code == 200:
            return r.json()
        if r.status_code in (429, 500, 502, 503):
            time.sleep(2 ** attempt); continue
        raise RuntimeError(f"YouTube API {r.status_code}: {r.text[:300]}")
    raise RuntimeError("YouTube API 재시도 초과")

def yt_uploads_playlist(url):
    m = re.search(r"youtube\.com/(@[^/?#]+)", url)
    if m:
        j = yt_get("channels", part="contentDetails,snippet", forHandle=m.group(1))
    else:
        m = re.search(r"youtube\.com/channel/(UC[\w\-]+)", url)
        if not m: raise ValueError(f"유튜브 채널 URL 형식을 인식하지 못했습니다: {url}")
        j = yt_get("channels", part="contentDetails,snippet", id=m.group(1))
    if not j.get("items"): raise ValueError(f"채널을 찾지 못했습니다: {url}")
    it = j["items"][0]
    return it["contentDetails"]["relatedPlaylists"]["uploads"], it["snippet"]["title"]

KW30 = re.compile(r"30\s*(th|周年|週年|주년|年)|30\s*anniv", re.I)

def yt_collect_videos(playlist_id):
    picked, token, stop = [], None, False
    while not stop and len(picked) < MAX_POSTS_PER_CHANNEL:
        p = {"part": "contentDetails,snippet", "playlistId": playlist_id, "maxResults": 50}
        if token: p["pageToken"] = token
        j = yt_get("playlistItems", **p)
        for it in j["items"]:
            pub = pd.to_datetime(it["contentDetails"].get("videoPublishedAt") or it["snippet"]["publishedAt"], utc=True).tz_localize(None)
            if pub < START_TS: stop = True; break                  # 업로드 목록은 최신순 → 이후는 모두 기간 이전
            if pub > END_TS + pd.Timedelta(days=1): continue
            picked.append((it["contentDetails"]["videoId"], pub))
            if len(picked) >= MAX_POSTS_PER_CHANNEL: break
        token = j.get("nextPageToken")
        if not token: break
    rows = []
    for i in range(0, len(picked), 50):
        batch = picked[i:i + 50]
        j = yt_get("videos", part="snippet,statistics", id=",".join(v for v, _ in batch))
        pub_map = dict(batch)
        for it in j["items"]:
            st, sn = it.get("statistics", {}), it["snippet"]
            text = (sn.get("title", "") + "\n" + sn.get("description", "")[:600]).strip()
            if ONLY_30TH_KEYWORD and not KW30.search(text): continue
            rows.append({"video_id": it["id"], "date": pub_map[it["id"]].strftime("%Y-%m-%d"), "copy": text,
                         "views": st.get("viewCount"), "likes": st.get("likeCount"), "comments": st.get("commentCount"),
                         "shares": 0, "url": f"https://www.youtube.com/watch?v={it['id']}"})
    return rows

def yt_collect_comments(video_id, n):
    out, token = [], None
    while len(out) < n:
        p = {"part": "snippet", "videoId": video_id, "maxResults": min(100, n - len(out)),
             "order": COMMENT_ORDER, "textFormat": "plainText"}
        if token: p["pageToken"] = token
        try:
            j = yt_get("commentThreads", **p)
        except RuntimeError as e:
            if "commentsDisabled" in str(e) or "403" in str(e) or "404" in str(e): break
            raise
        for it in j["items"]:
            s = it["snippet"]["topLevelComment"]["snippet"]
            out.append({"date": s["publishedAt"][:10], "text": s["textDisplay"]})
        token = j.get("nextPageToken")
        if not token: break
    return out

def collect_youtube(country):
    posts, cmts = [], []
    if not YOUTUBE_API_KEY:
        print(f"[{country}] YOUTUBE_API_KEY가 없어 YouTube 수집을 건너뜁니다."); return posts, cmts
    for _, ch in channels[(channels.country == country) & (channels.platform == "YouTube")].iterrows():
        try:
            pl, title = yt_uploads_playlist(ch["url"])
            vids = yt_collect_videos(pl)
        except Exception as e:
            print(f"[{country}] {ch['channel_name']} 실패: {e}"); continue
        print(f"[{country}] {ch['channel_name']}: 영상 {len(vids)}개")
        for v in vids:
            v.update({"country": country, "platform": "YouTube", "format": "영상", "post_key": v["url"],
                      "channel_name": ch["channel_name"], "channel_type": ch["channel_type"]})
            posts.append(v)
            for c in yt_collect_comments(v["video_id"], COMMENTS_PER_POST):
                cmts.append({**c, "country": country, "platform": "YouTube", "post_key": v["url"]})
            time.sleep(0.2)
    return posts, cmts

## 3. 빌리빌리 (yt-dlp · 실험적)

`yt-dlp`가 설치돼 있으면 채널(space) 페이지에서 영상 메타데이터와 댓글을 가져옵니다. 지역 제한·차단으로 실패할 수 있으며, 실패하면 `manual/중국_Bilibili_*.csv`로 대체하세요.

In [ ]:
def collect_bilibili(country):
    posts, cmts = [], []
    rows = channels[(channels.country == country) & (channels.platform == "Bilibili")]
    if rows.empty: return posts, cmts
    try:
        import yt_dlp
        from yt_dlp.utils import DateRange
    except ImportError:
        print(f"[{country}] yt-dlp 미설치 → 빌리빌리는 manual CSV로 입력하세요."); return posts, cmts
    for _, ch in rows.iterrows():
        url = ch["url"].split("?")[0].rstrip("/")
        if "space.bilibili.com" in url and not url.endswith("/video"): url += "/video"
        opts = {"quiet": True, "no_warnings": True, "skip_download": True, "ignoreerrors": True,
                "playlistend": MAX_POSTS_PER_CHANNEL, "sleep_interval": REQUEST_DELAY,
                "daterange": DateRange(START_TS.strftime("%Y%m%d"), END_TS.strftime("%Y%m%d")),
                "getcomments": COMMENTS_PER_POST > 0}
        try:
            with yt_dlp.YoutubeDL(opts) as ydl:
                info = ydl.extract_info(url, download=False)
        except Exception as e:
            print(f"[{country}] 빌리빌리 {ch['channel_name']} 실패: {str(e)[:100]}"); continue
        entries = [e for e in ((info or {}).get("entries") or []) if e]
        print(f"[{country}] 빌리빌리 {ch['channel_name']}: 영상 {len(entries)}개")
        for e in entries:
            d = e.get("upload_date") or ""
            text = ((e.get("title") or "") + "\n" + (e.get("description") or "")[:600]).strip()
            if ONLY_30TH_KEYWORD and not KW30.search(text): continue
            link = e.get("webpage_url") or e.get("original_url") or url
            posts.append({"country": country, "platform": "Bilibili", "format": "영상",
                          "date": f"{d[:4]}-{d[4:6]}-{d[6:8]}" if len(d) == 8 else "", "copy": text,
                          "views": e.get("view_count"), "likes": e.get("like_count"),
                          "comments": e.get("comment_count"), "shares": e.get("repost_count") or 0,
                          "url": link, "post_key": link, "channel_name": ch["channel_name"], "channel_type": ch["channel_type"]})
            for cm in (e.get("comments") or [])[:COMMENTS_PER_POST]:
                ts = cm.get("timestamp")
                cmts.append({"country": country, "platform": "Bilibili", "post_key": link, "text": cm.get("text", ""),
                             "date": pd.to_datetime(ts, unit="s").strftime("%Y-%m-%d") if ts else ""})
    return posts, cmts

## 4. 공식 홈페이지 수집

시작 URL(시트 3행 예시 URL + `HOMEPAGE_EXTRA`) → 같은 도메인 링크를 `CRAWL_DEPTH`단계까지 → **제목(h1~h3) 단위 콘텐츠 블록**으로 분해해 `본문 / CTA / 포맷 / 날짜`를 추출합니다.

In [ ]:
_ROBOTS = {}
def robots_ok(url):
    p = urlparse(url); base = f"{p.scheme}://{p.netloc}"
    if base not in _ROBOTS:
        rp = robotparser.RobotFileParser()
        try:
            r = requests.get(base + "/robots.txt", headers=UA, timeout=15)
            rp.parse(r.text.splitlines() if r.status_code == 200 else [])
        except Exception:
            print(f"⚠ {base}/robots.txt 를 읽지 못했습니다(허용으로 간주).")
            rp.parse([])
        _ROBOTS[base] = rp
    return _ROBOTS[base].can_fetch(UA["User-Agent"], url)

def fetch_html(url):
    if USE_PLAYWRIGHT:
        from playwright.sync_api import sync_playwright
        with sync_playwright() as pw:
            b = pw.chromium.launch(); pg = b.new_page(user_agent=UA["User-Agent"])
            pg.goto(url, wait_until="networkidle", timeout=60000); html = pg.content(); b.close()
        return html
    r = requests.get(url, headers=UA, timeout=30)
    r.raise_for_status(); r.encoding = r.apparent_encoding or r.encoding
    return r.text

CTA_HINT = re.compile(r"(예약|구매|신청|응모|자세히|바로가기|더보기|확인|참가|보러|예매|予約|購入|詳細|応募|申込|もっと|見る|チェック|参加|预约|购买|报名|详情|查看|了解|参与|立即|more|buy|shop|read|learn|view|join|enter|pre-?order|details|watch)", re.I)
DATE_RE = re.compile(r"(20\d{2})\s*[./\-年]\s*(\d{1,2})\s*[./\-月]\s*(\d{1,2})")
VIDEO_SRC = re.compile(r"youtube|youtu\.be|bilibili|vimeo|nicovideo|\.mp4", re.I)

def _date_in(text):
    m = DATE_RE.search(text or "")
    if not m: return ""
    try: return pd.Timestamp(int(m.group(1)), int(m.group(2)), int(m.group(3))).strftime("%Y-%m-%d")
    except Exception: return ""

def parse_blocks(url, html):
    soup = BeautifulSoup(html, "html.parser")
    title = (soup.title.get_text(" ", strip=True) if soup.title else "") or url
    meta = soup.find("meta", attrs={"property": "article:published_time"})
    tag = soup.find("time", attrs={"datetime": True})
    page_date = _date_in(meta["content"]) if meta and meta.get("content") else (_date_in(tag["datetime"]) if tag else "")
    for t in soup(["script", "style", "noscript", "svg", "nav", "footer"]): t.decompose()
    heads = soup.find_all(["h1", "h2", "h3"])
    blocks = []
    if not heads:
        heads = [None]
    for i, h in enumerate(heads):
        nxt = heads[i + 1] if i + 1 < len(heads) else None
        texts, imgs, vids, ctas = [], 0, 0, []
        nodes = (h.next_elements if h is not None else soup.body.descendants if soup.body else [])
        if h is not None: texts.append(h.get_text(" ", strip=True))
        total = 0
        for el in nodes:
            if nxt is not None and el is nxt: break
            if getattr(el, "name", None) == "img": imgs += 1
            elif getattr(el, "name", None) == "video": vids += 1
            elif getattr(el, "name", None) == "iframe" and VIDEO_SRC.search(el.get("src", "") or ""): vids += 1
            elif getattr(el, "name", None) in ("a", "button"):
                t = el.get_text(" ", strip=True)
                if t and len(t) <= 40 and (CTA_HINT.search(t) or el.name == "button"): ctas.append(t)
            elif type(el).__name__ == "NavigableString" and el.find_parent(["h1", "h2", "h3", "button"]) is None:
                s = el.strip()
                if s: texts.append(s); total += len(s)
            if total > 3000: break
        text = re.sub(r"\s+", " ", " ".join(texts)).strip()
        if len(text) < 15: continue
        fmt = "영상" if vids else ("복합" if imgs and len(text) > 40 else ("이미지" if imgs else "텍스트"))
        blocks.append({"menu": f"{title[:40]} › {(h.get_text(' ', strip=True)[:30] if h is not None else '본문')}",
                       "format": fmt, "copy": text[:1500], "cta": " | ".join(dict.fromkeys(ctas))[:200],
                       "date": _date_in(text) or page_date, "url": url})
    return blocks, soup

def collect_homepage(country):
    rows, seen_pages, queue = [], set(), [(u, 0) for u in hp_urls[country]]
    base_hosts = {urlparse(u).netloc for u, _ in queue}
    while queue and len(seen_pages) < MAX_PAGES_PER_SITE:
        url, depth = queue.pop(0)
        url = url.split("#")[0]
        if url in seen_pages: continue
        seen_pages.add(url)
        if not robots_ok(url):
            print(f"[{country}] robots.txt가 수집을 허용하지 않아 건너뜀: {url}"); continue
        try:
            html = fetch_html(url)
        except Exception as e:
            print(f"[{country}] 실패 {url}: {str(e)[:80]}"); continue
        blocks, soup = parse_blocks(url, html)
        rows += blocks
        if len(blocks) == 0:
            print(f"[{country}] 본문 블록이 없습니다(JS 렌더링 사이트일 수 있음 → USE_PLAYWRIGHT=True): {url}")
        if depth < CRAWL_DEPTH:
            for a in soup.find_all("a", href=True):
                link = urljoin(url, a["href"]).split("#")[0]
                if urlparse(link).netloc in base_hosts and link not in seen_pages and link.startswith("http"):
                    queue.append((link, depth + 1))
        time.sleep(REQUEST_DELAY)
    df = pd.DataFrame(rows).drop_duplicates(subset=["copy"]) if rows else pd.DataFrame(columns=["menu", "format", "copy", "cta", "date", "url"])
    df["country"] = country
    print(f"[{country}] 홈페이지 콘텐츠 블록 {len(df)}개 ({len(seen_pages)}페이지)")
    return df

## 5. 수동·반자동 가져오기 (Instagram · X · 웨이보 · 빌리빌리)

해당 플랫폼은 공식 API가 제한(인스타그램: 본인 비즈니스 계정만 / X: 유료 / 웨이보: 승인 필요)되어 있고 로그인 필요 페이지를 자동 수집하는 것은 약관 위반 소지가 큽니다. 아래 **템플릿 CSV**를 `manual/` 폴더에 두고, 공식 채널의 게시물을 보면서 채우거나(또는 승인된 도구·데이터 요청으로 확보한 값을 붙여넣어) 사용하세요. **분류 열(목적·메시지유형·감성·의도·토픽)은 비워 두면 자동 채움**, 값을 넣으면 그 값을 우선합니다.

- 게시물: `manual/{국가}_{플랫폼}_posts.csv` → `date, format, purpose, copy, views, likes, comments, shares, url, msg_type`
- 댓글: `manual/{국가}_{플랫폼}_comments.csv` → `post_url, date, text, sentiment, topic, intent` (`post_url`은 게시물 CSV의 `url`과 같게)

In [ ]:
POST_COLS = ["date", "format", "purpose", "copy", "views", "likes", "comments", "shares", "url", "msg_type"]
CMT_COLS_M = ["post_url", "date", "text", "sentiment", "topic", "intent"]

def manual_name(country, platform, kind):
    return os.path.join(MANUAL_DIR, f"{country}_{platform}_{kind}.csv")

def make_manual_templates():
    auto = {"YouTube"} if YOUTUBE_API_KEY else set()
    made = []
    for c in ORDER:
        for p in channels[channels.country == c]["platform"].dropna().unique():
            if p in auto: continue
            for kind, cols in (("posts", POST_COLS), ("comments", CMT_COLS_M)):
                f = manual_name(c, p, kind)
                if not os.path.exists(f):
                    pd.DataFrame(columns=cols).to_csv(f, index=False, encoding="utf-8-sig"); made.append(f)
    print("생성된 템플릿:" if made else "새로 만든 템플릿 없음", *made, sep="\n  ")

def norm_url(u):
    u = str(u).strip().lower()
    u = re.sub(r"^https?://(www\.|m\.|mobile\.)?", "", u)
    if "youtube.com/watch" in u:
        m = re.search(r"v=([\w\-]+)", u); return f"youtube.com/watch?v={m.group(1)}" if m else u
    return u.split("?")[0].split("#")[0].rstrip("/")

def load_manual(country):
    posts, cmts = [], []
    for p in channels[channels.country == country]["platform"].dropna().unique():
        fp, fc = manual_name(country, p, "posts"), manual_name(country, p, "comments")
        if os.path.exists(fp):
            d = pd.read_csv(fp, encoding="utf-8-sig", dtype=object)
            d = d[d[["copy", "date", "views"]].notna().any(axis=1)]
            for _, r in d.iterrows():
                posts.append({**{k: r.get(k) for k in POST_COLS}, "country": country, "platform": p, "post_key": r.get("url")})
        if os.path.exists(fc):
            d = pd.read_csv(fc, encoding="utf-8-sig", dtype=object)
            d = d[d["text"].notna()]
            for _, r in d.iterrows():
                cmts.append({**{k: r.get(k) for k in CMT_COLS_M if k != "post_url"}, "country": country, "platform": p, "post_key": r.get("post_url")})
    return posts, cmts

make_manual_templates()

# ② 통합 · 자동 분류 · CSV 저장

In [ ]:
def parse_count(x):
    if x is None or (isinstance(x, float) and np.isnan(x)) or pd.isna(x): return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)): return float(x)
    s = str(x).strip().replace(",", "").replace(" ", "")
    m = re.match(r"^([0-9]*\.?[0-9]+)([万萬만千천亿億억KkMmBb]?)", s)
    if not m: return np.nan
    u = {"万": 1e4, "萬": 1e4, "만": 1e4, "千": 1e3, "천": 1e3, "亿": 1e8, "億": 1e8, "억": 1e8, "K": 1e3, "k": 1e3, "M": 1e6, "m": 1e6, "B": 1e9, "b": 1e9}
    return float(m.group(1)) * u.get(m.group(2), 1)

hp_parts, post_rows, cmt_rows = [], [], []
for c in ORDER:
    hp_parts.append(collect_homepage(c))
    yp, yc = collect_youtube(c); bp, bc = collect_bilibili(c); mp, mc = load_manual(c)
    post_rows += yp + bp + mp; cmt_rows += yc + bc + mc

# ── 홈페이지 ──
hp = pd.concat(hp_parts, ignore_index=True) if hp_parts else pd.DataFrame()
if len(hp):
    hp["date"] = pd.to_datetime(hp["date"], errors="coerce")
    hp = hp.sort_values(["country", "date"], na_position="last").reset_index(drop=True)
    hp["post_no"] = hp.groupby("country").cumcount() + 1
    hp["post_id"] = [f"HP_{CC[c]}_{n:03d}" for c, n in zip(hp["country"], hp["post_no"])]
    r = hp["copy"].map(classify_purpose); hp["purpose"] = [x[0] for x in r]; f1 = [x[1] for x in r]
    for k in PROPOSAL_KW: hp[k] = hp["copy"].map(lambda t, k=k: classify_proposals(t)[k])
    r = hp["copy"].map(classify_msg_type); hp["msg_type"] = [x[0] for x in r]; f2 = [x[1] for x in r]
    hp["review_flag"] = [("목적" if a else "") + (" 메시지유형" if b else "") for a, b in zip(f1, f2)]
    hp["date"] = hp["date"].dt.strftime("%Y-%m-%d")
    hp = hp[["country", "post_no", "post_id", "date", "menu", "format", "purpose", "copy", "cta",
             "p_collect", "p_battle", "p_memory", "p_event", "msg_type", "url", "review_flag"]]

# ── SNS 게시물 ──
posts = pd.DataFrame(post_rows)
cmts = pd.DataFrame(cmt_rows)
if len(posts):
    posts["date"] = pd.to_datetime(posts["date"], errors="coerce")
    posts = posts[(posts["date"].isna()) | ((posts["date"] >= START_TS) & (posts["date"] <= END_TS + pd.Timedelta(days=1)))]
    posts = posts.sort_values(["country", "date"], na_position="last").reset_index(drop=True)
    posts["post_no"] = posts.groupby("country").cumcount() + 1
    posts["post_id"] = [f"SNS_{CC[c]}_{n:03d}" for c, n in zip(posts["country"], posts["post_no"])]
    for col in ["views", "likes", "comments", "shares"]: posts[col] = posts[col].map(parse_count)
    posts["shares"] = posts["shares"].fillna(0)
    posts["key"] = posts["post_key"].map(norm_url)
    # 수동 입력값이 있으면 우선, 없으면 자동 분류
    pr = posts["copy"].map(classify_purpose)
    posts["purpose"] = [p if isinstance(p, str) and p.strip() else a[0] for p, a in zip(posts.get("purpose", pd.Series([None] * len(posts))), pr)]
    mt = posts["copy"].map(classify_msg_type)
    posts["msg_type"] = [m if isinstance(m, str) and m.strip() else a[0] for m, a in zip(posts.get("msg_type", pd.Series([None] * len(posts))), mt)]
    posts["review_flag"] = [("목적" if a[1] else "") + (" 메시지유형" if b[1] else "") for a, b in zip(pr, mt)]
    posts["date"] = posts["date"].dt.strftime("%Y-%m-%d")
    for col in ["channel_name", "channel_type"]:
        if col not in posts: posts[col] = ""
    key2no = {(c, k): n for c, k, n in zip(posts["country"], posts["key"], posts["post_no"])}
    posts_out = posts[["country", "post_no", "post_id", "platform", "date", "format", "purpose", "copy", "views", "likes",
                       "comments", "shares", "msg_type", "url", "channel_name", "channel_type", "review_flag"]]
else:
    key2no, posts_out = {}, pd.DataFrame()

# ── 댓글 ──
if len(cmts):
    cmts["post_no"] = [key2no.get((c, norm_url(k))) for c, k in zip(cmts["country"], cmts["post_key"])]
    miss = cmts["post_no"].isna().sum()
    if miss: print(f"⚠ 게시물과 연결되지 않은 댓글 {miss}건 (post_url 확인)")
    cmts["date"] = pd.to_datetime(cmts["date"], errors="coerce").dt.strftime("%Y-%m-%d")
    auto_s = classify_sentiment_batch(cmts["text"].tolist())
    man_s = pd.to_numeric(cmts.get("sentiment"), errors="coerce") if "sentiment" in cmts else pd.Series([np.nan] * len(cmts))
    cmts["sentiment"] = [m if m in (-1, 0, 1) else a for m, a in zip(man_s, auto_s)]
    cmts["intent"] = [i if str(i).strip().upper() in ("O", "X") else classify_intent(t)
                      for i, t in zip(cmts.get("intent", pd.Series([None] * len(cmts))), cmts["text"])]
    cmts["topic"] = [t if isinstance(t, str) and t.strip() else classify_topics(x)
                     for t, x in zip(cmts.get("topic", pd.Series([None] * len(cmts))), cmts["text"])]
    cmts_out = cmts[["country", "platform", "post_no", "date", "text", "sentiment", "topic", "intent"]]
else:
    cmts_out = pd.DataFrame()

hp.to_csv(f"{CSV_DIR}/hp_all.csv", index=False, encoding="utf-8-sig")
posts_out.to_csv(f"{CSV_DIR}/sns_posts.csv", index=False, encoding="utf-8-sig")
cmts_out.to_csv(f"{CSV_DIR}/comments.csv", index=False, encoding="utf-8-sig")
print(f"저장 완료 → {CSV_DIR}/  홈페이지 {len(hp)}건 | SNS 게시물 {len(posts_out)}건 | 댓글 {len(cmts_out)}건")
for name, d in [("홈페이지", hp), ("SNS", posts_out), ("댓글", cmts_out)]:
    if len(d): print(f"\n[{name}] 국가별 건수:", d["country"].value_counts().reindex(ORDER).fillna(0).astype(int).to_dict())

# ③ 원본 xlsx 사본에 채우기

- 원본을 `수집결과/`로 **복사한 뒤** 4행부터 기입 (2행 입력가이드·3행 작성예시는 그대로 둠)
- 수집값 + **자동 분류값**(목적·제안 O/X·메시지유형·감성·토픽·의도)이 함께 들어가며, 비고에 `자동분류(1차)`와 검수 필요 항목이 표시됩니다
- 번호(ID) 수식은 국가 코드에 맞게(`SNS_JP_`, `SNS_CN_` …) 다시 쓰고, 500행을 넘으면 이어서 확장
- 댓글 시트의 「매칭 게시물 번호」는 SNS 게시물 시트 번호와 자동으로 맞춤
- 홈페이지에서 게시일을 못 찾은 블록은 **노출일 = 수집일**로 적고 비고에 표시

In [ ]:
PLAT_KO = {"YouTube": "유튜브", "Instagram": "인스타그램", "X": "X", "Weibo": "웨이보", "Bilibili": "빌리빌리"}
CMT_PFX = {"YouTube": "YTB", "Instagram": "INST", "X": "X", "Weibo": "WB", "Bilibili": "BILI"}

def to_int(x):
    if x is None or pd.isna(x): return None
    if isinstance(x, (int, float, np.integer, np.floating)): return int(x)
    s = str(x).strip().replace(",", "").replace(" ", "")
    m = re.match(r"^([0-9]*\.?[0-9]+)([万萬만千천亿億억KkMmBb]?)", s)
    if not m: return None
    u = {"万": 1e4, "萬": 1e4, "만": 1e4, "千": 1e3, "천": 1e3, "亿": 1e8, "億": 1e8, "억": 1e8, "K": 1e3, "k": 1e3, "M": 1e6, "m": 1e6, "B": 1e9, "b": 1e9}
    return int(round(float(m.group(1)) * u.get(m.group(2), 1)))

def clean_cell(v):
    if v is None or (isinstance(v, float) and np.isnan(v)): return None
    if isinstance(v, str):
        v = ILLEGAL_CHARACTERS_RE.sub("", v).strip()
        return v[:32000] if v else None
    return v

def put(ws, r, c, v, date_fmt=False):
    v = clean_cell(v)
    if v is None: return
    cell = ws.cell(r, c)
    if date_fmt and isinstance(v, str):
        try: v = pd.Timestamp(v).to_pydatetime(); cell.number_format = "yyyy-mm-dd"
        except Exception: pass
    cell.value = v
    if isinstance(v, str) and v.startswith("="): cell.data_type = "s"        # '=)' 같은 댓글이 수식으로 읽히지 않게

def write_formulas(ws, kind, c, n_rows, plat=None):
    for r in range(4, max(ws.max_row, 3 + n_rows) + 1):
        if kind == "hp":
            ws.cell(r, 1).value = f'=IF(B{r}="","","HP_{CC[c]}_"&TEXT(ROW()-3,"000"))'
            ws.cell(r, 2).value = f'=IF(C{r}="","","{c}")'
        elif kind == "sns":
            ws.cell(r, 1).value = f'=IF(B{r}="","","SNS_{CC[c]}_"&TEXT(ROW()-3,"000"))'
        else:
            ws.cell(r, 1).value = f'=IF(B{r}="","","{CMT_PFX[plat]}_{CC[c]}_"&TEXT(ROW()-3,"000"))'

def note_join(*parts):
    return " · ".join(p for p in parts if p)

def flag_note(r):
    f = r.get("review_flag"); f = f.strip() if isinstance(f, str) else ""
    return "자동분류(1차)" + (f" 검수필요: {f}" if f else "")

def empty_like(df, cols):
    return df if len(df) else pd.DataFrame(columns=cols)

def fill_sheets():
    log = []
    for c in ORDER:
        # ── 홈페이지 시트 ──
        src = find_file(c, "홈페이지"); dst = os.path.join(XLSX_OUT_DIR, os.path.basename(src)); shutil.copy2(src, dst)
        wb = openpyxl.load_workbook(dst); ws = wb.worksheets[0]
        h = hp[hp["country"] == c].sort_values("post_no") if len(hp) else pd.DataFrame()
        write_formulas(ws, "hp", c, len(h))
        for _, r in h.iterrows():
            row = 3 + int(r["post_no"]); d = r.get("date"); nodate = not (isinstance(d, str) and d)
            put(ws, row, 3, TODAY if nodate else d, date_fmt=True)
            for col, k in ((4, "menu"), (5, "format"), (6, "purpose"), (7, "copy"), (8, "cta"),
                           (9, "p_collect"), (10, "p_battle"), (11, "p_memory"), (12, "p_event"), (13, "msg_type"), (14, "url")):
                put(ws, row, col, r.get(k))
            put(ws, row, 15, note_join("게시일 미확인→노출일=수집일" if nodate else "", flag_note(r)))
        wb.save(dst); log.append((c, "홈페이지", len(h)))

        # ── SNS 시트 ──
        src = find_file(c, "SNS"); dst = os.path.join(XLSX_OUT_DIR, os.path.basename(src)); shutil.copy2(src, dst)
        wb = openpyxl.load_workbook(dst)
        p = posts[posts["country"] == c].sort_values("post_no") if len(posts) else pd.DataFrame()
        main = next(w for w in wb.worksheets if "정량" in nfc(w.title))
        write_formulas(main, "sns", c, len(p))
        for _, r in p.iterrows():
            row = 3 + int(r["post_no"])
            put(main, row, 2, PLAT_KO.get(r["platform"], r["platform"]))
            put(main, row, 3, r.get("date") if isinstance(r.get("date"), str) else None, date_fmt=True)
            put(main, row, 4, r.get("format")); put(main, row, 5, r.get("purpose")); put(main, row, 6, r.get("copy"))
            for col, k in ((7, "views"), (8, "likes"), (9, "comments"), (10, "shares")):
                put(main, row, col, to_int(r.get(k)))
            put(main, row, 11, r.get("msg_type")); put(main, row, 12, r.get("url"))
            ch = " / ".join(str(x) for x in (r.get("channel_name"), r.get("channel_type")) if isinstance(x, str) and x)
            put(main, row, 13, note_join(ch, "공유수 미제공(0)" if r["platform"] == "YouTube" else "", flag_note(r)))
        log.append((c, "SNS 게시물", len(p)))

        cm = cmts[cmts["country"] == c] if len(cmts) else pd.DataFrame()
        for w in wb.worksheets:
            if "댓글" not in nfc(w.title): continue
            plat = norm_platform(nfc(w.title).replace("댓글 모음", ""))
            d = cm[cm["platform"] == plat].sort_values(["post_no", "date"], na_position="last").reset_index(drop=True) if len(cm) else pd.DataFrame()
            if plat in CMT_PFX: write_formulas(w, "cmt", c, len(d), plat)
            for i, r in d.iterrows():
                row = 4 + i; no = r.get("post_no")
                put(w, row, 2, f"SNS_{CC[c]}_{int(no):03d}" if pd.notna(no) else None)
                put(w, row, 3, r.get("date") if isinstance(r.get("date"), str) else None, date_fmt=True)
                put(w, row, 4, r.get("text"))
                s = r.get("sentiment"); put(w, row, 5, int(s) if pd.notna(s) else None)
                put(w, row, 6, r.get("topic")); put(w, row, 7, r.get("intent"))
                if pd.isna(no): put(w, row, 8, "게시물 번호 매칭 실패(링크 확인)")
            log.append((c, f"댓글-{plat}", len(d)))
        wb.save(dst)
    print("시트 채우기 완료 → 폴더:", os.path.abspath(XLSX_OUT_DIR))
    print(pd.DataFrame(log, columns=["국가", "시트", "기입 행 수"]).pivot_table(index="시트", columns="국가", values="기입 행 수", aggfunc="sum").reindex(columns=ORDER).fillna(0).astype(int))

if WRITE_SHEETS:
    fill_sheets()

# ④ 분석 (표 3개 + 그래프 10개)

`data/hp_all.csv`, `data/sns_posts.csv`, `data/comments.csv`를 읽습니다. (CSV를 직접 고쳐서 다시 이 섹션부터 실행해도 됩니다. 분류 값은 `O/X`, 감성은 `-1/0/1`)

In [ ]:
import os, re, unicodedata, warnings
from collections import Counter
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
from scipy import stats
from IPython.display import display

warnings.filterwarnings("ignore")

# ───────── 설정 ─────────
FIG_DIR  = os.path.join("결과", "figures")   # 그래프 PNG 저장 폴더
OUT_XLSX = os.path.join("결과", "분석결과_통합.xlsx")
os.makedirs(FIG_DIR, exist_ok=True)

ORDER  = ["한국", "중국", "일본"]
CC     = {"한국": "KR", "중국": "CN", "일본": "JP"}
COLORS = {"한국": "#3B6FD4", "중국": "#D64545", "일본": "#2F9E6E"}

# ───────── 한글(+한자·가나) 폰트: 없으면 자동 설치 (재시작 불필요) ─────────
FONT_CANDS = ["Malgun Gothic", "AppleGothic", "NanumGothic", "NanumBarunGothic", "Noto Sans CJK KR",
              "Noto Sans KR", "Noto Sans CJK JP", "Noto Sans CJK SC", "Source Han Sans KR"]

def installed_cjk():
    have = {f.name for f in font_manager.fontManager.ttflist}
    return [c for c in FONT_CANDS if c in have]

def set_font():
    import sys, glob, subprocess
    use = installed_cjk()
    if not use:        # 1) pip 패키지(폰트 동봉): Colab·Windows·Mac 공통, 재시작 불필요
        try:
            try:
                import koreanize_matplotlib
            except ImportError:
                subprocess.run([sys.executable, "-m", "pip", "install", "-q", "koreanize-matplotlib"], check=True)
                import koreanize_matplotlib
            use = installed_cjk()
        except Exception as e:
            print("koreanize-matplotlib 설치 실패:", str(e)[:80])
    if not use:        # 2) 리눅스(Colab 등): apt로 나눔·Noto CJK 설치 후 바로 등록
        try:
            subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum", "fonts-noto-cjk"], check=True, capture_output=True)
            for f in glob.glob("/usr/share/fonts/**/*.[to]t[fc]", recursive=True):
                if "Nanum" in f or "NotoSansCJK" in f:
                    font_manager.fontManager.addfont(f)
            use = installed_cjk()
        except Exception as e:
            print("apt 폰트 설치 실패:", str(e)[:80])
    if use:
        mpl.rcParams["font.family"] = "sans-serif"
        mpl.rcParams["font.sans-serif"] = use + [f for f in mpl.rcParams["font.sans-serif"] if f not in use]
        print("사용 폰트:", use[0])
    else:
        print("⚠ 한글 폰트를 설치하지 못했습니다. 셀에서 `!pip install koreanize-matplotlib` 실행 → 런타임 재시작 → 처음부터 다시 실행하세요.")
    mpl.rcParams["axes.unicode_minus"] = False

set_font()
# 폰트 확인용 (한글·일본어·중국어·기호가 네모로 보이면 위 안내대로 설치)
_fig, _ax = plt.subplots(figsize=(6.5, 0.7)); _ax.axis("off")
_ax.text(0, 0.5, "한글 테스트 · 日本語 · 中文 · ± × % – ›", fontsize=14, va="center"); plt.show()
mpl.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200,
                     "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 220)
pd.options.display.float_format = "{:,.2f}".format

In [ ]:
def nfc(s):
    return unicodedata.normalize("NFC", s)

def find_file(country, keyword):
    for fn in sorted(os.listdir(DATA_DIR)):
        n = nfc(fn)
        if n.lower().endswith(".xlsx") and not n.startswith("~$") and n.startswith(country) and keyword in n:
            return os.path.join(DATA_DIR, fn)
    raise FileNotFoundError(f"'{country}' + '{keyword}' 파일을 {os.path.abspath(DATA_DIR)} 에서 찾지 못했습니다.")

def read_all_sheets(path):
    xl = pd.ExcelFile(path)
    return {nfc(s): xl.parse(s, header=0, dtype=object) for s in xl.sheet_names}

# ── 값 정제 함수 ──
def clean_text(s):
    s = s.where(s.isna(), s.astype(str).str.strip())
    return s.replace({"": np.nan, "nan": np.nan, "NaT": np.nan})

_UNIT = {"万": 1e4, "萬": 1e4, "만": 1e4, "千": 1e3, "천": 1e3, "亿": 1e8, "億": 1e8, "억": 1e8,
         "K": 1e3, "k": 1e3, "M": 1e6, "m": 1e6, "B": 1e9, "b": 1e9}
def parse_count(x):
    if pd.isna(x): return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)): return float(x)
    s = str(x).strip().replace(",", "").replace(" ", "")
    m = re.match(r"^([0-9]*\.?[0-9]+)([万萬만千천亿億억KkMmBb]?)", s)
    if not m: return np.nan
    return float(m.group(1)) * _UNIT.get(m.group(2), 1)

def parse_one_date(x):
    if pd.isna(x): return pd.NaT
    if isinstance(x, (int, float, np.integer, np.floating)) and 20000 < float(x) < 80000:
        return pd.Timestamp("1899-12-30") + pd.Timedelta(days=float(x))      # 엑셀 시리얼
    return pd.to_datetime(x, errors="coerce")
def parse_date(s):
    return pd.to_datetime(s.apply(parse_one_date), errors="coerce")

def parse_ox(s):
    m = {"O": 1, "○": 1, "ㅇ": 1, "Y": 1, "YES": 1, "1": 1, "1.0": 1,
         "X": 0, "×": 0, "N": 0, "NO": 0, "0": 0, "0.0": 0}
    return s.astype(str).str.strip().str.upper().map(m)

def parse_sent(s):
    num = pd.to_numeric(s, errors="coerce")
    num = num.where(num.isin([-1, 0, 1]))
    txt = s.astype(str).str.strip().str.lower().map({"긍정": 1, "중립": 0, "부정": -1,
                                                    "positive": 1, "neutral": 0, "negative": -1})
    return num.fillna(txt)

def norm_format(x):
    if pd.isna(x): return np.nan
    s = str(x).strip()
    return "복합" if any(k in s for k in ["+", "복합", "/"]) else s

def norm_platform(x):
    s = str(x).strip().lower()
    if s in ("", "nan"): return np.nan
    if any(k in s for k in ["유튜브", "youtube", "油管"]): return "YouTube"
    if any(k in s for k in ["인스타", "instagram", "insta"]): return "Instagram"
    if any(k in s for k in ["웨이보", "weibo", "微博"]): return "Weibo"
    if any(k in s for k in ["빌리빌리", "bilibili", "哔哩", "b站"]): return "Bilibili"
    if s == "x" or "트위터" in s or "twitter" in s or s.startswith("x(") or s.startswith("x ("): return "X"
    return str(x).strip()

# 국가 간 비교를 위한 플랫폼군 (분석자 가정: 필요하면 수정)
PLAT_GROUP = {"YouTube": "영상형", "Bilibili": "영상형", "Instagram": "이미지형", "X": "단문형", "Weibo": "단문형"}

In [ ]:
IN_DIR = CSV_DIR
HP_NEED  = ["country", "post_no", "post_id", "date", "menu", "format", "purpose", "copy", "cta",
            "p_collect", "p_battle", "p_memory", "p_event", "msg_type", "url"]
SNS_NEED = ["country", "post_no", "post_id", "platform", "date", "format", "purpose", "copy",
            "views", "likes", "comments", "shares", "msg_type", "url"]
CMT_NEED = ["country", "platform", "post_no", "date", "text", "sentiment", "topic", "intent"]

def read_csv(name, need):
    path = os.path.join(IN_DIR, name)
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} 이 없습니다. 먼저 01_수집_분류 노트북을 실행하세요.")
    try:
        d = pd.read_csv(path, encoding="utf-8-sig")
    except pd.errors.EmptyDataError:
        d = pd.DataFrame()
    d = d.reindex(columns=need + [c for c in d.columns if c not in need])
    if d.empty:                                          # 빈 CSV면 열 자료형이 float로 깨지므로 object로 고정
        d = pd.DataFrame({c: pd.Series(dtype=object) for c in d.columns})
    return d

hp_all  = read_csv("hp_all.csv", HP_NEED)
sns_all = read_csv("sns_posts.csv", SNS_NEED)
cmt_all = read_csv("comments.csv", CMT_NEED)

# ── 자료형 정리 ──
for c in ["p_collect", "p_battle", "p_memory", "p_event"]:
    hp_all[c] = parse_ox(hp_all[c])
hp_all["date"] = parse_date(hp_all["date"])

for c in ["views", "likes", "comments", "shares"]:
    sns_all[c] = sns_all[c].map(parse_count)
sns_all["post_no"] = pd.to_numeric(sns_all["post_no"], errors="coerce")
sns_all["platform"] = sns_all["platform"].map(norm_platform)
sns_all["date"] = parse_date(sns_all["date"])
sns_all["plat_group"] = sns_all["platform"].map(PLAT_GROUP)
sns_all["interactions"] = sns_all[["likes", "comments", "shares"]].sum(axis=1, min_count=1)
sns_all["er"] = np.where(sns_all["views"] > 0, sns_all["interactions"] / sns_all["views"] * 100, np.nan)
sns_all["month"] = sns_all["date"].dt.to_period("M").dt.to_timestamp()

cmt_all["post_no"] = pd.to_numeric(cmt_all["post_no"], errors="coerce")
cmt_all["platform"] = cmt_all["platform"].map(norm_platform)
cmt_all["date"] = parse_date(cmt_all["date"])
cmt_all["sentiment"] = parse_sent(cmt_all["sentiment"])
cmt_all["intent"] = parse_ox(cmt_all["intent"])

if hp_all.empty and sns_all.empty and cmt_all.empty:
    raise ValueError("CSV에 데이터가 없습니다. 01_수집_분류 노트북의 수집 결과를 확인하세요.")

# 댓글 → 게시물 단위 집계를 SNS 게시물에 결합 (국가 + 게시물 번호)
post_cmt = (cmt_all.dropna(subset=["post_no"]).groupby(["country", "post_no"])
            .agg(n_cmt=("text", "size"), mean_sent=("sentiment", "mean"), intent_rate=("intent", "mean")).reset_index())
sns_all = sns_all.merge(post_cmt, on=["country", "post_no"], how="left")

SENT_LABEL = {-1: "부정", 0: "중립", 1: "긍정"}
cmt_all["sent_label"] = cmt_all["sentiment"].map(SENT_LABEL)
cmt_all["plat_label"] = cmt_all["country"].astype(str) + "·" + cmt_all["platform"].astype(str)

print(f"홈페이지 {len(hp_all):,}건 | SNS 게시물 {len(sns_all):,}건 | 댓글·반응 {len(cmt_all):,}건")
if sns_all.empty:
    print("⚠ SNS 게시물이 0건입니다 → YOUTUBE_API_KEY 설정, 채널 URL, manual/*_posts.csv 입력 여부를 확인하세요. (그래프 4~7은 비어 나옵니다)")
if cmt_all.empty:
    print("⚠ 댓글이 0건입니다 → YOUTUBE_API_KEY 설정, COMMENTS_PER_POST > 0, manual/*_comments.csv 입력 여부를 확인하세요. (그래프 8~10과 감성·행동의도 분석은 비어 나옵니다)")
miss = cmt_all.dropna(subset=["post_no"]).merge(sns_all[["country", "post_no"]].assign(ok=1), on=["country", "post_no"], how="left")["ok"].isna().sum()
if miss: print(f"⚠ 게시물과 연결되지 않은 댓글 {miss:,}건")

## 3. 표 1 — 데이터 현황

In [ ]:
def period(s):
    s = s.dropna()
    return "-" if s.empty else f"{s.min():%Y-%m-%d} ~ {s.max():%Y-%m-%d}"

t1 = pd.DataFrame(index=ORDER)
t1.index.name = "국가"
t1["홈페이지 콘텐츠(건)"] = hp_all.groupby("country").size().reindex(ORDER).fillna(0).astype(int)
t1["SNS 게시물(건)"]     = sns_all.groupby("country").size().reindex(ORDER).fillna(0).astype(int)
t1["댓글·반응(건)"]      = cmt_all.groupby("country").size().reindex(ORDER).fillna(0).astype(int)
plat_cnt = (cmt_all.groupby(["country", "platform"]).size().unstack(fill_value=0).reindex(ORDER).fillna(0).astype(int))
plat_cnt.columns = [f"댓글-{c}" for c in plat_cnt.columns]
t1 = t1.join(plat_cnt).fillna(0)
t1["수집 기간(SNS)"] = [period(sns_all.loc[sns_all.country == c, "date"]) for c in ORDER]
print("[표 1] 국가별 데이터 현황")
display(t1)

## 4. 홈페이지 콘텐츠 분석 (그래프 1~3)

In [ ]:
def save_show(fig, name):
    fig.savefig(os.path.join(FIG_DIR, f"{name}.png"), bbox_inches="tight")
    plt.show()

def no_data(ax, msg="데이터 없음"):
    ax.text(0.5, 0.5, msg, ha="center", va="center", transform=ax.transAxes, color="gray")
    ax.set_xticks([]); ax.set_yticks([])

def pct_table(df, col, cols_order=None, idx="country", idx_order=None):
    idx_order = idx_order or ORDER
    ct = pd.crosstab(df[idx], df[col]).reindex(idx_order).fillna(0)
    if cols_order:
        ct = ct[[c for c in cols_order if c in ct.columns] + [c for c in ct.columns if c not in cols_order]]
    n = ct.sum(axis=1)
    return ct.div(n.replace(0, np.nan), axis=0) * 100, n

def stacked_barh(ax, p, n, title, colors=None):
    if p.shape[1] == 0 or n.sum() == 0:
        no_data(ax); ax.set_title(title, fontsize=11); return
    pal = colors or list(plt.get_cmap("Set2").colors)
    left = np.zeros(len(p)); y = np.arange(len(p))
    for i, col in enumerate(p.columns):
        v = p[col].fillna(0).values
        ax.barh(y, v, left=left, label=str(col), color=pal[i % len(pal)], edgecolor="white")
        for yy, (l, w) in enumerate(zip(left, v)):
            if w >= 7: ax.text(l + w / 2, yy, f"{w:.0f}%", ha="center", va="center", fontsize=9)
        left += v
    ax.set_yticks(y); ax.set_yticklabels([f"{i}\n(n={int(n[i])})" for i in p.index])
    ax.set_xlim(0, 100); ax.invert_yaxis(); ax.set_xlabel("비율(%)"); ax.set_title(title, fontsize=11)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=min(len(p.columns), 4), frameon=False, fontsize=9)

def grouped_bar(ax, df, cat, val, title, ylabel, cat_order=None, fmt="{:.2f}"):
    d = df.dropna(subset=[cat, val])
    if d.empty:
        no_data(ax); ax.set_title(title, fontsize=11); return
    present = list(d[cat].unique())
    cats = ([c for c in cat_order if c in present] + [c for c in present if c not in (cat_order or [])]) if cat_order else sorted(present)
    g = d.groupby(["country", cat])[val].agg(["mean", "sem"]).reset_index()
    w = 0.8 / len(ORDER); x = np.arange(len(cats))
    for k, c in enumerate(ORDER):
        gc = g[g.country == c].set_index(cat).reindex(cats)
        m = gc["mean"].values; e = gc["sem"].fillna(0).values
        ax.bar(x + (k - 1) * w, np.nan_to_num(m), w, yerr=e, capsize=2, color=COLORS[c], label=c, alpha=.9)
        for xx, mm in zip(x + (k - 1) * w, m):
            if not np.isnan(mm): ax.text(xx, mm, fmt.format(mm), ha="center", va="bottom", fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(cats); ax.set_ylabel(ylabel); ax.set_title(title, fontsize=11)
    ax.legend(frameon=False, fontsize=9)

# ── 그래프 1. 홈페이지 콘텐츠 포맷 · 캠페인 목적 구성 ──
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
p, n = pct_table(hp_all, "format", ["텍스트", "이미지", "영상", "복합"])
stacked_barh(axes[0], p, n, "홈페이지 콘텐츠 포맷 구성")
p, n = pct_table(hp_all, "purpose", ["상품판매", "행사참여", "브랜드홍보"])
stacked_barh(axes[1], p, n, "홈페이지 캠페인 목적 구성")
fig.suptitle("그래프 1. 국가별 홈페이지 콘텐츠 포맷·캠페인 목적", fontsize=13, y=1.02)
save_show(fig, "fig01_homepage_format_purpose")

In [ ]:
# ── 그래프 2. 홈페이지 제안 유형(수집·배틀·추억·행사) 포함 비율 ──
PROP = {"p_collect": "수집", "p_battle": "배틀", "p_memory": "추억", "p_event": "행사"}
rows = []
for c in ORDER:
    h = hp_all[hp_all.country == c]
    for k, v in PROP.items():
        s = h[k].dropna()
        rows.append({"country": c, "제안 유형": v, "포함비율(%)": s.mean() * 100 if len(s) else np.nan})
prop_df = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(8.5, 4.2))
grouped_bar(ax, prop_df, "제안 유형", "포함비율(%)", "그래프 2. 국가별 홈페이지 제안 유형 포함 비율",
            "해당 제안을 포함한 콘텐츠 비율(%)", cat_order=list(PROP.values()), fmt="{:.0f}")
ax.set_ylim(0, 105)
save_show(fig, "fig02_homepage_proposal_types")

In [ ]:
# ── 그래프 3. 메시지 유형(내부자형 vs 초대형): 홈페이지 vs SNS ──
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
MSG = ["내부자형", "초대형"]
mcol = ["#8E7CC3", "#F2A541"]
p, n = pct_table(hp_all, "msg_type", MSG);  stacked_barh(axes[0], p, n, "홈페이지", colors=mcol)
p, n = pct_table(sns_all, "msg_type", MSG); stacked_barh(axes[1], p, n, "SNS 게시물", colors=mcol)
fig.suptitle("그래프 3. 국가별 메시지 유형(내부자형·초대형) 구성", fontsize=13, y=1.02)
save_show(fig, "fig03_message_type")

## 5. SNS 정량 반응 분석 (그래프 4~7)

- **참여율(%)** = (좋아요 + 댓글 + 공유) ÷ 조회수 × 100 (조회수가 0/결측이면 제외)
- **총반응수** = 좋아요 + 댓글 + 공유  — 국가·플랫폼별 조회수 정의가 달라 규모 비교는 참고용으로만 사용

In [ ]:
# ── 그래프 4. 국가별 참여율·총반응수 분포 ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
rng = np.random.default_rng(0)

def box_by_country(ax, col, title, ylabel, log10=False):
    data = []
    for c in ORDER:
        v = sns_all.loc[sns_all.country == c, col].dropna()
        if log10: v = np.log10(v + 1)
        data.append(v.values)
    if all(len(d) == 0 for d in data):
        no_data(ax); ax.set_title(title, fontsize=11); return
    bp = ax.boxplot([d if len(d) else [np.nan] for d in data], patch_artist=True, widths=.5, showfliers=False)
    for patch, c in zip(bp["boxes"], ORDER):
        patch.set_facecolor(COLORS[c]); patch.set_alpha(.35)
    for med in bp["medians"]: med.set_color("black")
    for i, (d, c) in enumerate(zip(data, ORDER), start=1):
        ax.scatter(i + rng.uniform(-.15, .15, len(d)), d, s=14, color=COLORS[c], alpha=.7, zorder=3)
    ax.set_xticks(range(1, len(ORDER) + 1)); ax.set_xticklabels([f"{c}\n(n={len(d)})" for c, d in zip(ORDER, data)])
    ax.set_ylabel(ylabel); ax.set_title(title, fontsize=11)

box_by_country(axes[0], "er", "참여율(%)", "참여율(%)")
box_by_country(axes[1], "interactions", "총반응수 (로그)", "log10(총반응수+1)", log10=True)
fig.suptitle("그래프 4. 국가별 SNS 게시물 반응 분포", fontsize=13, y=1.02)
save_show(fig, "fig04_engagement_distribution")

In [ ]:
# ── 그래프 5. 캠페인 목적별 · 메시지 유형별 평균 참여율 ──
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
grouped_bar(axes[0], sns_all, "purpose", "er", "캠페인 목적별", "평균 참여율(%)  ±SEM",
            cat_order=["상품판매", "행사참여", "브랜드홍보"])
grouped_bar(axes[1], sns_all, "msg_type", "er", "메시지 유형별", "평균 참여율(%)  ±SEM", cat_order=MSG)
fig.suptitle("그래프 5. 콘텐츠 전략별 SNS 평균 참여율", fontsize=13, y=1.02)
save_show(fig, "fig05_engagement_by_strategy")

In [ ]:
# ── 그래프 6. 플랫폼 × 국가 평균 참여율 히트맵 ──
piv = sns_all.pivot_table(index="platform", columns="country", values="er", aggfunc="mean").reindex(columns=ORDER)
cnt = sns_all.pivot_table(index="platform", columns="country", values="post_id", aggfunc="count").reindex(columns=ORDER)
fig, ax = plt.subplots(figsize=(7.5, 1.2 + 0.9 * max(len(piv), 1)))
if piv.dropna(how="all").empty:
    no_data(ax)
else:
    cmap = plt.get_cmap("YlOrRd").copy(); cmap.set_bad("#eeeeee")
    im = ax.imshow(np.ma.masked_invalid(piv.values.astype(float)), cmap=cmap, aspect="auto")
    ax.set_xticks(range(len(ORDER))); ax.set_xticklabels(ORDER)
    ax.set_yticks(range(len(piv))); ax.set_yticklabels(piv.index)
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            v = piv.iloc[i, j]; k = cnt.iloc[i, j]
            ax.text(j, i, "–" if np.isnan(v) else f"{v:.2f}%\n(n={int(k)})", ha="center", va="center", fontsize=9)
    fig.colorbar(im, ax=ax, label="평균 참여율(%)")
ax.set_title("그래프 6. 플랫폼 × 국가 평균 참여율", fontsize=12)
save_show(fig, "fig06_platform_heatmap")

In [ ]:
# ── 그래프 7. 월별 게시 추이와 평균 참여율 ──
d = sns_all.dropna(subset=["month"])
mc = d.pivot_table(index="month", columns="country", values="post_id", aggfunc="count").reindex(columns=ORDER)
me = d.pivot_table(index="month", columns="country", values="er", aggfunc="mean").reindex(columns=ORDER)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, tb, ttl, yl in [(axes[0], mc, "월별 게시물 수", "게시물 수(건)"), (axes[1], me, "월별 평균 참여율", "평균 참여율(%)")]:
    if tb.dropna(how="all").empty:
        no_data(ax)
    else:
        for c in ORDER:
            ax.plot(tb.index, tb[c], marker="o", color=COLORS[c], label=c)
        ax.legend(frameon=False); ax.set_ylabel(yl)
        ax.tick_params(axis="x", rotation=45)
    ax.set_title(ttl, fontsize=11)
fig.suptitle("그래프 7. 30주년 기간 SNS 게시·반응 추이", fontsize=13, y=1.02)
save_show(fig, "fig07_monthly_trend")

## 6. 댓글 감성 · 행동의도 분석 (그래프 8~10)

감성: 1=긍정, 0=중립, -1=부정 (시트의 사전코딩 값) / 행동의도: O=구매·시작 의도 표현

In [ ]:
# ── 그래프 8. 댓글 감성 분포 (국가별 · 국가×플랫폼별) ──
SENT_ORDER = ["부정", "중립", "긍정"]
scol = ["#D64545", "#BDBDBD", "#3B6FD4"]
pl_order = (cmt_all[["country", "plat_label"]].drop_duplicates()
            .assign(o=lambda x: x["country"].map(ORDER.index)).sort_values(["o", "plat_label"])["plat_label"].tolist())

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), gridspec_kw={"width_ratios": [1, 1.25]})
p, n = pct_table(cmt_all, "sent_label", SENT_ORDER)
stacked_barh(axes[0], p, n, "국가별", colors=scol)
if pl_order:
    p, n = pct_table(cmt_all, "sent_label", SENT_ORDER, idx="plat_label", idx_order=pl_order)
    stacked_barh(axes[1], p, n, "국가·플랫폼별", colors=scol)
else:
    no_data(axes[1])
fig.suptitle("그래프 8. 팬 댓글 감성 분포", fontsize=13, y=1.02)
save_show(fig, "fig08_sentiment")

In [ ]:
# ── 그래프 9. 행동·참여 의도 비율 (국가별 · 감성별) ──
def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n)
    a = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return (c - a) / d, (c + a) / d

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
ax = axes[0]
ps, los, his, ns = [], [], [], []
for c in ORDER:
    v = cmt_all.loc[cmt_all.country == c, "intent"].dropna()
    k, n_ = v.sum(), len(v); lo, hi = wilson(k, n_)
    ps.append(k / n_ * 100 if n_ else np.nan); los.append(lo * 100 if n_ else np.nan); his.append(hi * 100 if n_ else np.nan); ns.append(n_)
ps, los, his = map(np.array, (ps, los, his))
if np.isnan(ps).all():
    no_data(ax)
else:
    ax.bar(ORDER, np.nan_to_num(ps), color=[COLORS[c] for c in ORDER], alpha=.9,
           yerr=[np.nan_to_num(ps - los), np.nan_to_num(his - ps)], capsize=4)
    for i, (pp, nn) in enumerate(zip(ps, ns)):
        if not np.isnan(pp): ax.text(i, pp, f"{pp:.1f}%\n(n={nn})", ha="center", va="bottom", fontsize=9)
    ax.set_ylabel("행동·참여 의도(O) 비율(%)")
ax.set_title("국가별 (95% Wilson 신뢰구간)", fontsize=11)

cmt_all["intent_pct"] = cmt_all["intent"] * 100
grouped_bar(axes[1], cmt_all, "sent_label", "intent_pct", "감성별 행동의도 비율",
            "행동·참여 의도(O) 비율(%)", cat_order=SENT_ORDER, fmt="{:.0f}")
fig.suptitle("그래프 9. 팬 댓글의 행동·참여 의도", fontsize=13, y=1.02)
save_show(fig, "fig09_intent")

In [ ]:
# ── 그래프 10. 국가별 주요 언급 토픽 Top 10 ──
def split_topics(s):
    if pd.isna(s): return []
    return [t.strip() for t in re.split(r"[/,，、;；·|\n]+", str(s)) if t.strip()]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
topic_tables = {}
for ax, c in zip(axes, ORDER):
    sub = cmt_all[cmt_all.country == c]
    cnt_ = Counter(t for s in sub["topic"] for t in split_topics(s))
    topic_tables[c] = pd.DataFrame(cnt_.most_common(30), columns=["토픽", "언급수"])
    top = cnt_.most_common(10)
    if not top:
        no_data(ax)
    else:
        labels, vals = zip(*top)
        ax.barh(range(len(top)), vals, color=COLORS[c], alpha=.9)
        ax.set_yticks(range(len(top))); ax.set_yticklabels(labels); ax.invert_yaxis()
        for i, v in enumerate(vals): ax.text(v, i, f" {v}", va="center", fontsize=9)
        ax.set_xlabel("언급 수")
    ax.set_title(f"{c} (댓글 {len(sub):,}건)", fontsize=11)
fig.suptitle("그래프 10. 국가별 주요 언급 토픽 Top 10", fontsize=13, y=1.02)
fig.tight_layout()
save_show(fig, "fig10_topics")

## 7. 표 2 — 핵심 지표 비교 · 표 3 — 통계 검정

In [ ]:
def kpi(c):
    h = hp_all[hp_all.country == c]; s = sns_all[sns_all.country == c]; m = cmt_all[cmt_all.country == c]
    sv = m["sentiment"].dropna(); iv = m["intent"].dropna()
    return {
        "홈페이지 콘텐츠(건)": len(h),
        "SNS 게시물(건)": len(s),
        "댓글·반응(건)": len(m),
        "평균 조회수": s["views"].mean(),
        "중앙값 조회수": s["views"].median(),
        "평균 좋아요": s["likes"].mean(),
        "평균 댓글수": s["comments"].mean(),
        "평균 공유수": s["shares"].mean(),
        "평균 참여율(%)": s["er"].mean(),
        "중앙값 참여율(%)": s["er"].median(),
        "댓글 평균 감성(-1~1)": sv.mean(),
        "긍정 비율(%)": (sv == 1).mean() * 100 if len(sv) else np.nan,
        "부정 비율(%)": (sv == -1).mean() * 100 if len(sv) else np.nan,
        "행동의도(O) 비율(%)": iv.mean() * 100 if len(iv) else np.nan,
        "초대형 비율-홈페이지(%)": (h["msg_type"].dropna() == "초대형").mean() * 100 if h["msg_type"].notna().any() else np.nan,
        "초대형 비율-SNS(%)": (s["msg_type"].dropna() == "초대형").mean() * 100 if s["msg_type"].notna().any() else np.nan,
    }

t2 = pd.DataFrame({c: kpi(c) for c in ORDER}).astype(object)
t2.iloc[:3] = t2.iloc[:3].astype(int)       # 건수는 정수로 표시
t2.index.name = "지표"
print("[표 2] 국가별 핵심 지표 비교")
display(t2)

In [ ]:
def stars(p):
    if pd.isna(p): return "-"
    return "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else "n.s."

def fmt_p(p):
    return "-" if pd.isna(p) else ("<.001" if p < .001 else f"{p:.3f}")

def empty_row(label, test, note):
    return {"분석": label, "검정": test, "통계량": np.nan, "자유도": np.nan, "p값": "-", "효과크기": np.nan, "유의": "-", "비고": note}

def chi2_row(df, col, label):
    d = df[["country", col]].dropna()
    ct = pd.crosstab(d["country"], d[col])
    ct = ct.loc[ct.sum(axis=1) > 0, ct.sum(axis=0) > 0]
    test = "카이제곱 (효과크기=Cramér's V)"
    if ct.shape[0] < 2 or ct.shape[1] < 2:
        return empty_row(label, test, "표본·범주 부족")
    chi2, p, dof, exp = stats.chi2_contingency(ct)
    v = np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))
    return {"분석": label, "검정": test, "통계량": chi2, "자유도": dof, "p값": fmt_p(p), "효과크기": v,
            "유의": stars(p), "비고": "기대빈도<5 셀 있음(해석 주의)" if (exp < 5).any() else ""}

def kw_rows(df, col, label):
    groups = {c: df.loc[df.country == c, col].dropna().values for c in ORDER}
    groups = {c: g for c, g in groups.items() if len(g) >= 2}
    test = "Kruskal-Wallis (효과크기=ε²)"
    if len(groups) < 2:
        return [empty_row(label, test, "비교 가능한 국가가 2개 미만")]
    try:
        H, p = stats.kruskal(*groups.values())
    except ValueError:
        return [empty_row(label, test, "모든 값이 동일")]
    n = sum(len(g) for g in groups.values()); k = len(groups)
    rows = [{"분석": label, "검정": test, "통계량": H, "자유도": k - 1, "p값": fmt_p(p),
             "효과크기": (H - k + 1) / (n - k) if n > k else np.nan, "유의": stars(p), "비고": ""}]
    pairs = list(combinations(groups, 2))
    for a, b in pairs:
        U, pp = stats.mannwhitneyu(groups[a], groups[b], alternative="two-sided")
        padj = min(1.0, pp * len(pairs))
        rows.append({"분석": f"  └ 사후: {a} vs {b}", "검정": "Mann-Whitney U (Bonferroni 보정, 효과크기=rank-biserial r)",
                     "통계량": U, "자유도": np.nan, "p값": fmt_p(padj),
                     "효과크기": 2 * U / (len(groups[a]) * len(groups[b])) - 1, "유의": stars(padj),
                     "비고": f"r>0 이면 {a} 쪽이 더 큼"})
    return rows

rows = []
# 홈페이지
rows.append(chi2_row(hp_all, "format", "홈페이지 콘텐츠 포맷 × 국가"))
rows.append(chi2_row(hp_all, "purpose", "홈페이지 캠페인 목적 × 국가"))
rows.append(chi2_row(hp_all, "msg_type", "홈페이지 메시지 유형 × 국가"))
for k, v in PROP.items():
    r = chi2_row(hp_all, k, f"홈페이지 '{v}' 제안 포함 여부 × 국가")
    r["비고"] = (r["비고"] + " / 제안 4종 동시 검정(다중비교 주의)").strip(" /")
    rows.append(r)
# SNS
rows += kw_rows(sns_all, "er", "SNS 참여율(%) 국가 간 차이")
rows += kw_rows(sns_all, "interactions", "SNS 총반응수 국가 간 차이")
rows.append(chi2_row(sns_all, "purpose", "SNS 캠페인 목적 × 국가"))
rows.append(chi2_row(sns_all, "msg_type", "SNS 메시지 유형 × 국가"))
rows.append(chi2_row(sns_all, "format", "SNS 게시물 포맷 × 국가"))
# 댓글
rows.append(chi2_row(cmt_all, "sentiment", "댓글 감성(-1/0/1) × 국가"))
rows.append(chi2_row(cmt_all, "intent", "행동의도(O/X) × 국가"))
# 게시물 단위: 참여율 ↔ 댓글 평균 감성
for c in ORDER:
    d = sns_all.loc[sns_all.country == c, ["er", "mean_sent"]].dropna()
    label = f"[{c}] 게시물 참여율 ↔ 댓글 평균 감성"
    if len(d) >= 5 and d["er"].nunique() > 1 and d["mean_sent"].nunique() > 1:
        rho, p = stats.spearmanr(d["er"], d["mean_sent"])
        rows.append({"분석": label, "검정": "Spearman 상관 (효과크기=ρ)", "통계량": rho, "자유도": len(d) - 2,
                     "p값": fmt_p(p), "효과크기": rho, "유의": stars(p), "비고": f"n={len(d)} 게시물"})
    else:
        rows.append(empty_row(label, "Spearman 상관", f"유효 게시물 {len(d)}개(5개 이상 필요)"))

t3 = pd.DataFrame(rows)
print("[표 3] 통계 검정 결과  (* p<.05, ** p<.01, *** p<.001, n.s.=비유의)")
display(t3)

## 8. 저장

In [ ]:
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE
def xl_clean(df):
    f = lambda v: ILLEGAL_CHARACTERS_RE.sub("", v)[:32000] if isinstance(v, str) else v     # 엑셀이 허용하지 않는 제어문자 제거
    return df.apply(lambda col: col.map(f))

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as xw:
    t1.to_excel(xw, sheet_name="표1_데이터현황")
    t2.to_excel(xw, sheet_name="표2_핵심지표")
    t3.to_excel(xw, sheet_name="표3_통계검정", index=False)
    xl_clean(hp_all).to_excel(xw, sheet_name="통합_홈페이지", index=False)
    xl_clean(sns_all).to_excel(xw, sheet_name="통합_SNS게시물", index=False)
    xl_clean(cmt_all.drop(columns=["intent_pct"], errors="ignore")).to_excel(xw, sheet_name="통합_댓글", index=False)
    for c, tb in topic_tables.items():
        xl_clean(tb).to_excel(xw, sheet_name=f"토픽_{c}", index=False)
print(f"저장 완료 → {OUT_XLSX}, {FIG_DIR}/ (그래프 PNG 10개)")

### 해석 시 참고
- **참여율**은 조회수가 없는 게시물(웨이보 등)에서 계산되지 않습니다. 이 경우 `총반응수`·`표3`의 총반응수 검정을 참고하세요.
- **플랫폼군 비교**(그래프 6)는 YouTube↔Bilibili, X↔Weibo를 대응 플랫폼으로 본 가정입니다. 중국에는 Instagram이 없어 해당 칸은 비게 됩니다.
- 감성·행동의도는 **사전코딩 값**을 그대로 사용합니다. 논문에는 2인 이상 코딩 후 일치도(Cohen's κ)를 함께 보고하는 것이 안전합니다.
- 표본이 비확률 표집(공식 채널 게시물·상위 댓글 등)이면 검정 결과는 **탐색적 비교**로 서술하세요.

# ⑤ 검수 표본 · Cohen's κ

자동 분류의 타당도 보고용입니다. ① `data/review_sample_*.csv` 생성 → ② 사람이 `human_*` 열을 채움(가능하면 2인) → ③ `kappa_report()`로 일치도 계산

In [ ]:
def make_review_sample(n_comments=150, n_posts=40, n_hp=40, seed=42):
    out = {}
    if len(cmts_out):
        s = cmts_out.groupby("country", group_keys=False).apply(lambda g: g.sample(min(len(g), n_comments // 3), random_state=seed)).copy()
        s["human_sentiment"] = ""; s["human_intent"] = ""; out["comments"] = s
    if len(posts_out):
        s = posts_out.groupby("country", group_keys=False).apply(lambda g: g.sample(min(len(g), n_posts // 3 + 1), random_state=seed)).copy()
        s["human_purpose"] = ""; s["human_msg_type"] = ""; out["posts"] = s
    if len(hp):
        s = hp.groupby("country", group_keys=False).apply(lambda g: g.sample(min(len(g), n_hp // 3 + 1), random_state=seed)).copy()
        s["human_purpose"] = ""; s["human_msg_type"] = ""; out["hp"] = s
    for k, d in out.items():
        d.to_csv(f"{CSV_DIR}/review_sample_{k}.csv", index=False, encoding="utf-8-sig")
    print("검수 표본 저장:", {k: len(v) for k, v in out.items()})

def cohen_kappa(a, b):
    a, b = pd.Series(list(a)), pd.Series(list(b))
    m = a.notna() & b.notna() & (a.astype(str) != "") & (b.astype(str) != "")
    a, b = a[m].astype(str), b[m].astype(str)
    if len(a) == 0: return np.nan
    cats = sorted(set(a) | set(b)); po = (a == b).mean()
    pe = sum((a == c).mean() * (b == c).mean() for c in cats)
    return (po - pe) / (1 - pe) if pe < 1 else np.nan

def kappa_report():
    pairs = [("comments", "sentiment", "human_sentiment"), ("comments", "intent", "human_intent"),
             ("posts", "purpose", "human_purpose"), ("posts", "msg_type", "human_msg_type"),
             ("hp", "purpose", "human_purpose"), ("hp", "msg_type", "human_msg_type")]
    rows = []
    for k, auto, human in pairs:
        f = f"{CSV_DIR}/review_sample_{k}.csv"
        if not os.path.exists(f): continue
        d = pd.read_csv(f, encoding="utf-8-sig")
        if human not in d or d[human].isna().all(): continue
        rows.append({"대상": k, "항목": auto, "n": int(d[human].notna().sum()),
                     "일치율": (d[auto].astype(str) == d[human].astype(str)).mean(), "Cohen's κ": cohen_kappa(d[auto], d[human])})
    return pd.DataFrame(rows)

make_review_sample()
# 사람이 human_* 열을 채운 뒤:  kappa_report()